# 📦 Notebook 3: Denormalization

When indexes aren't enough, denormalization trades storage for speed by eliminating expensive joins.

## Learning Objectives

By the end of this notebook, you'll understand:
- Normalized vs denormalized schemas
- When to denormalize
- Materialized views
- Pre-computed aggregations

---

🔍 **Open Adminer** at http://localhost:8080 to see table structures and run queries!

## 🛠️ Setup

**1. Start PostgreSQL + Redis + visualization tools** (from the lab root):

```bash
cd 04-patterns/scaling-reads
docker compose up -d
uv sync
```

**2. Select the `.venv` kernel**

Click the kernel picker in the top-right of this notebook and choose the
`.venv` Python interpreter. If it doesn't appear, reload the VS Code window
(`Cmd+Shift+P` → "Developer: Reload Window") and try again.

### 🔍 Visualization tools (optional but recommended)

| Tool | URL | Use it to |
|------|-----|-----------|
| Adminer (PostgreSQL) | http://localhost:8080 | See tables, run SQL, view execution plans |
| RedisInsight | http://localhost:5540 | Watch cache keys, TTLs, hits/misses |

**Adminer login**: System `PostgreSQL`, Server `postgres`, User `demo`,
Password `demo`, Database `scaling_demo`.


In [ ]:
import psycopg2
import time

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "scaling_demo",
    "user": "demo",
    "password": "demo"
}

def get_connection():
    return psycopg2.connect(**DB_CONFIG)

def run_query(query: str):
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute(query)
    try:
        results = cursor.fetchall()
    except:
        results = []
    conn.commit()
    conn.close()
    return results

def measure_query(query: str) -> tuple:
    conn = get_connection()
    cursor = conn.cursor()
    start = time.time()
    cursor.execute(query)
    results = cursor.fetchall()
    elapsed = (time.time() - start) * 1000
    conn.close()
    return elapsed, results

print("✅ Connected to PostgreSQL")

## 📊 Normalized vs Denormalized

**Normalization** eliminates data redundancy by splitting data across tables. **Denormalization** adds redundancy back for faster reads.

In [ ]:
print("📊 Normalized vs Denormalized Schema")
print("=" * 60)
print("""
NORMALIZED (Our current schema)
─────────────────────────────────────────────────────────────
┌──────────┐     ┌──────────┐     ┌──────────┐
│  users   │     │  posts   │     │ comments │
├──────────┤     ├──────────┤     ├──────────┤
│ id       │◄────│ user_id  │◄────│ post_id  │
│ username │     │ content  │     │ user_id  │
│ email    │     │ likes    │     │ content  │
└──────────┘     └──────────┘     └──────────┘

To get a post with author info:
SELECT p.*, u.username FROM posts p JOIN users u ON p.user_id = u.id

─────────────────────────────────────────────────────────────

DENORMALIZED (Redundant but fast)
─────────────────────────────────────────────────────────────
┌─────────────────────────────┐
│       feed_items            │
├─────────────────────────────┤
│ post_id                     │
│ post_content                │
│ author_username  ◄─ COPY!   │
│ author_avatar    ◄─ COPY!   │
│ like_count                  │
└─────────────────────────────┘

To get a post with author info:
SELECT * FROM feed_items WHERE post_id = 123

─────────────────────────────────────────────────────────────
""")

print("💡 Denormalization eliminates JOINs by storing redundant data!")

## 🔬 The Cost of Joins

In [ ]:
print("🔬 Normalized Query: Feed with joins")
print("=" * 60)

normalized_query = """
SELECT 
    p.id, p.content, p.like_count, p.created_at,
    u.username, u.display_name, u.profile_image_url
FROM posts p
JOIN users u ON p.user_id = u.id
ORDER BY p.created_at DESC
LIMIT 20
"""

normalized_ms = min(measure_query(normalized_query)[0] for _ in range(3))
elapsed, results = measure_query(normalized_query)
print(f"\nTime: {normalized_ms:.2f}ms (best of 3) | Rows: {len(results)}")

print("\nSample result:")
if results:
    row = results[0]
    print(f"  Post ID: {row[0]}")
    print(f"  Content: {row[1][:50]}...")
    print(f"  Likes: {row[2]}")
    print(f"  Author: {row[4]}")

In [ ]:
print("\n🔨 Creating denormalized feed_items table...")

run_query("DELETE FROM feed_items")

run_query("""
INSERT INTO feed_items (
    viewer_user_id, post_id, author_user_id,
    author_username, author_display_name, author_profile_image,
    post_content, post_image_url, like_count, comment_count, created_at
)
SELECT 
    1,  -- For demo, all items are for user 1
    p.id, p.user_id,
    u.username, u.display_name, u.profile_image_url,
    p.content, p.image_url, p.like_count, p.comment_count, p.created_at
FROM posts p
JOIN users u ON p.user_id = u.id
""")

run_query("CREATE INDEX IF NOT EXISTS idx_feed_viewer ON feed_items(viewer_user_id, created_at DESC)")

print("✅ Feed items created!")

In [ ]:
print("🔬 Denormalized Query: Feed without joins")
print("=" * 60)

denormalized_query = """
SELECT 
    post_id, post_content, like_count, created_at,
    author_username, author_display_name, author_profile_image
FROM feed_items
WHERE viewer_user_id = 1
ORDER BY created_at DESC
LIMIT 20
"""

denormalized_ms = min(measure_query(denormalized_query)[0] for _ in range(3))
elapsed, results = measure_query(denormalized_query)
print(f"\nTime: {denormalized_ms:.2f}ms (best of 3) | Rows: {len(results)}")

print(f"\n📊 normalized (JOIN) {normalized_ms:.2f}ms  vs  denormalized "
      f"{denormalized_ms:.2f}ms  →  {normalized_ms / denormalized_ms:.2f}x")
print("\n💡 No JOINs needed - all data is in one table!")
print()
print("⚖️  Be suspicious of that ratio. At LIMIT 20 with an index on created_at,")
print("    the join costs 20 primary-key lookups — Postgres is very good at those,")
print("    so the win here is small or even negative. Denormalization pays when")
print("    the join FANS OUT: a feed assembled from 500 followees, an N+1 that")
print("    becomes 500 round trips, or a join key with no index at all.")
print("    If you can't measure the win, you're paying the write cost for nothing.")

assert len(results) == 20, f"expected a 20-row page, got {len(results)}"

## ⚖️ Trade-offs

In [ ]:
print("⚖️ Denormalization Trade-offs")
print("=" * 60)
print("""
PROS:
─────────────────────────────────────────────────────────────
✅ Faster reads (no JOINs)
✅ Simpler queries
✅ Predictable performance
✅ Easier to cache

CONS:
─────────────────────────────────────────────────────────────
❌ More storage used
❌ Writes become complex (update multiple places)
❌ Data can become inconsistent
❌ Schema changes are harder

WHEN TO DENORMALIZE:
─────────────────────────────────────────────────────────────
• Read/write ratio > 100:1
• JOINs are killing performance
• Source data changes infrequently
• You can tolerate brief staleness

EXAMPLES:
─────────────────────────────────────────────────────────────
• Social media feeds (denormalize author info)
• E-commerce orders (denormalize product names)
• Analytics dashboards (denormalize everything!)
""")

### 🔀 The bill for denormalization: copies drift

"Data can become inconsistent" is easy to write in a trade-off list and easy to
forget. Let's actually make it happen: rename a user, then ask the same question
two ways.


In [ ]:
print("🔀 Demonstrating the cost: denormalized copies drift")
print("=" * 60)

# Pick an author who actually appears in the feed we just built.
author_id, old_name = measure_query("""
    SELECT author_user_id, author_username
    FROM feed_items
    WHERE viewer_user_id = 1
    ORDER BY created_at DESC
    LIMIT 1
""")[1][0]

copies = measure_query(
    f"SELECT COUNT(*) FROM feed_items WHERE author_user_id = {author_id}"
)[1][0][0]
print(f"\nAuthor {author_id} is currently '{old_name}', copied into "
      f"{copies} feed_items row(s).")

# Rename them — a single-row UPDATE against the source of truth.
new_name = f"renamed{int(time.time()) % 100000}"
assert new_name != old_name, "pick a genuinely new name or this proves nothing"
run_query(f"UPDATE users SET username = '{new_name}' WHERE id = {author_id}")
print(f"\n✏️  UPDATE users SET username = '{new_name}' WHERE id = {author_id}")

joined = measure_query(f"""
    SELECT u.username FROM posts p JOIN users u ON p.user_id = u.id
    WHERE u.id = {author_id} LIMIT 1
""")[1][0][0]
copied = measure_query(
    f"SELECT author_username FROM feed_items WHERE author_user_id = {author_id} LIMIT 1"
)[1][0][0]

print(f"\n   normalized JOIN says : {joined!r}")
print(f"   feed_items copy says : {copied!r}")

assert joined == new_name, "the join must reflect the source of truth immediately"
assert copied == old_name, (
    f"the denormalized copy was supposed to go stale, but it says {copied!r} — "
    f"something else is already keeping feed_items in sync"
)
print("   ❌ Two answers to the same question. The JOIN cannot be wrong; the copy can.")

# The fix is not clever. It is just MORE WRITES.
run_query(
    f"UPDATE feed_items SET author_username = '{new_name}' "
    f"WHERE author_user_id = {author_id}"
)
copied = measure_query(
    f"SELECT author_username FROM feed_items WHERE author_user_id = {author_id} LIMIT 1"
)[1][0][0]
assert copied == new_name
print(f"\n   ✅ Fixed by fanning that one rename out to all {copies} copies.")
print(f"      1 logical write → {copies + 1} physical writes. THAT is the")
print( "      denormalization bill, and it is charged on every single update.")
print()
print("   In production that fan-out is a background job or a CDC stream, which")
print("   means it is also ASYNCHRONOUS — so there is always a window where the")
print("   feed shows the old name. Same shape as replication lag (Notebook 4)")
print("   and cache staleness (Notebook 5): a copy that hasn't caught up yet.")

## 📊 Materialized Views

Materialized views are like cached query results that PostgreSQL manages for you.

In [ ]:
print("📊 Creating Materialized View for Product Ratings")
print("=" * 60)

run_query("DROP MATERIALIZED VIEW IF EXISTS product_avg_ratings")

run_query("""
CREATE MATERIALIZED VIEW product_avg_ratings AS
SELECT 
    p.id,
    p.name,
    p.category,
    COALESCE(AVG(r.rating), 0) as avg_rating,
    COUNT(r.id) as review_count
FROM products p
LEFT JOIN reviews r ON p.id = r.product_id
GROUP BY p.id, p.name, p.category
""")

run_query("CREATE INDEX idx_product_ratings_category ON product_avg_ratings(category)")

print("✅ Materialized view created!")

In [ ]:
print("\n🔬 Query Comparison: Regular vs Materialized View")
print("=" * 60)

regular_query = """
SELECT 
    p.id, p.name, AVG(r.rating) as avg_rating, COUNT(r.id)
FROM products p
LEFT JOIN reviews r ON p.id = r.product_id
WHERE p.category = 'Electronics'
GROUP BY p.id, p.name
ORDER BY avg_rating DESC
LIMIT 10
"""

materialized_query = """
SELECT id, name, avg_rating, review_count
FROM product_avg_ratings
WHERE category = 'Electronics'
ORDER BY avg_rating DESC
LIMIT 10
"""

# Best of 3 each — a single sample on a laptop is noise, and a speedup number
# nobody re-measures is the first thing to rot.
elapsed1 = min(measure_query(regular_query)[0] for _ in range(3))
elapsed2 = min(measure_query(materialized_query)[0] for _ in range(3))
_, results1 = measure_query(regular_query)
_, results2 = measure_query(materialized_query)

print(f"\nRegular query (JOIN + GROUP BY): {elapsed1:.2f}ms")
print(f"Materialized view query:          {elapsed2:.2f}ms")
print(f"\nSpeedup: {elapsed1/elapsed2:.1f}x faster!")

assert len(results1) == len(results2) == 10, (
    f"both queries should return a 10-row page, got {len(results1)} and {len(results2)}"
)
assert elapsed1 > elapsed2, (
    f"the materialized view was not faster than the live aggregate "
    f"({elapsed2:.2f}ms vs {elapsed1:.2f}ms) — with no aggregation left to do "
    f"it should be, so something is off"
)
print("\n💡 The view does no work at read time: the GROUP BY over 20,000 reviews")
print("   already happened, once, at REFRESH time. You moved the cost, you")
print("   didn't remove it — and you now owe a refresh schedule.")

In [ ]:
print("\n🔄 Refreshing Materialized Views")
print("=" * 60)
print("""
Materialized views are SNAPSHOTS - they don't auto-update!

REFRESH OPTIONS:
─────────────────────────────────────────────────────────────
1. Manual refresh (blocks reads during refresh):
   REFRESH MATERIALIZED VIEW product_avg_ratings;

2. Concurrent refresh (allows reads during refresh):
   REFRESH MATERIALIZED VIEW CONCURRENTLY product_avg_ratings;
   (Requires unique index on view)

3. Scheduled refresh via cron or background job:
   - Every 5 minutes for frequently changing data
   - Every hour for slowly changing data
   - Nightly for analytics views
""")

print("\n🔄 Refreshing our view...")
run_query("REFRESH MATERIALIZED VIEW product_avg_ratings")
print("✅ View refreshed!")

### 🕰️ Watch a materialized view go stale

The quiz below says a materialized view goes stale "immediately after source
data changes". That's a claim, so let's make it happen and watch: read the
rating two ways, insert one review, and read again.


In [ ]:
print("🕰️ A materialized view is a SNAPSHOT, not a query")
print("=" * 60)

PRODUCT_ID = 1

# Repeatable: clear out any review this cell left behind on a previous run.
run_query("DELETE FROM reviews WHERE title = 'stale demo'")
run_query("REFRESH MATERIALIZED VIEW product_avg_ratings")


def live_rating(product_id: int) -> tuple:
    """The truth, computed from the reviews table right now."""
    rows = measure_query(f"""
        SELECT ROUND(COALESCE(AVG(r.rating), 0), 4), COUNT(r.id)
        FROM products p LEFT JOIN reviews r ON p.id = r.product_id
        WHERE p.id = {product_id}
        GROUP BY p.id
    """)[1]
    return (float(rows[0][0]), rows[0][1])


def view_rating(product_id: int) -> tuple:
    """Whatever the view captured the last time it was refreshed."""
    rows = measure_query(
        f"SELECT ROUND(avg_rating, 4), review_count "
        f"FROM product_avg_ratings WHERE id = {product_id}"
    )[1]
    return (float(rows[0][0]), rows[0][1])


before_live, before_view = live_rating(PRODUCT_ID), view_rating(PRODUCT_ID)
print(f"\n1. Straight after REFRESH")
print(f"   live table : avg={before_live[0]:.4f} over {before_live[1]} reviews")
print(f"   the view   : avg={before_view[0]:.4f} over {before_view[1]} reviews  ✅ agree")
assert before_live == before_view, (
    f"a freshly refreshed view must match its source: {before_live} vs {before_view}"
)

# A user leaves a scathing review.
run_query(
    f"INSERT INTO reviews (product_id, user_id, rating, title, content) "
    f"VALUES ({PRODUCT_ID}, 1, 1, 'stale demo', 'one star')"
)
print("\n2. A user posts a 1-star review...")

after_live, still_view = live_rating(PRODUCT_ID), view_rating(PRODUCT_ID)
print(f"   live table : avg={after_live[0]:.4f} over {after_live[1]} reviews  ← moved")
print(f"   the view   : avg={still_view[0]:.4f} over {still_view[1]} reviews  ← did NOT")

assert after_live != before_live, "the live aggregate must reflect the new review"
assert still_view == before_view, (
    f"the view is supposed to be frozen until REFRESH, but it changed: "
    f"{before_view} -> {still_view}"
)
assert still_view != after_live
print("\n   ❌ Every product page reading the view is now showing a wrong rating,")
print("      and will keep doing so until someone refreshes. Nothing errors.")

run_query("REFRESH MATERIALIZED VIEW product_avg_ratings")
caught_up = view_rating(PRODUCT_ID)
print(f"\n3. After REFRESH MATERIALIZED VIEW")
print(f"   the view   : avg={caught_up[0]:.4f} over {caught_up[1]} reviews  ✅ caught up")
assert caught_up == after_live

# Leave the table as we found it.
run_query("DELETE FROM reviews WHERE title = 'stale demo'")
run_query("REFRESH MATERIALIZED VIEW product_avg_ratings")

print()
print("👉 So the real design question is never 'should I materialize?' but")
print("   'how stale may this be?'. Your refresh interval IS your staleness SLA.")
print("   Note also that a plain REFRESH takes an ACCESS EXCLUSIVE lock — every")
print("   reader blocks for its duration. CONCURRENTLY avoids that, but needs a")
print("   UNIQUE index on the view:")
print("       CREATE UNIQUE INDEX ON product_avg_ratings (id);")
print("       REFRESH MATERIALIZED VIEW CONCURRENTLY product_avg_ratings;")

## 🧪 Quick Quiz

1. **When should you denormalize?**

2. **What's the downside of denormalization?**

3. **When do materialized views become stale?**

In [ ]:
print("📝 Quiz Answers")
print("=" * 50)
print()
print("1. When to denormalize:")
print("   - High read/write ratio (>100:1)")
print("   - JOINs are bottleneck")
print("   - Data changes infrequently")
print("   - Can tolerate brief staleness")
print()
print("2. Downside of denormalization:")
print("   - Data duplication (storage cost)")
print("   - Write complexity (update multiple places)")
print("   - Potential inconsistency")
print("   - Harder schema changes")
print()
print("3. Materialized views become stale:")
print("   - Immediately after source data changes")
print("   - They're snapshots, not live queries")
print("   - Must explicitly REFRESH")

## 📚 Summary

### Key Takeaways

1. **Denormalization trades writes for reads** - duplicate data for faster queries
2. **Use for high read/write ratios** - when reads dominate
3. **Materialized views** - database-managed cached aggregations
4. **Your refresh interval IS your staleness SLA** - a materialized view is\n   wrong from the instant the source changes until the next REFRESH
5. **Not a silver bullet** - adds complexity to writes

### Next Up

In **Notebook 4**, we'll learn about read replicas:
- Leader-follower replication
- Handling replication lag
- Scaling beyond single server